# NHS A&E External Feature Enrichment

## Goal
This notebook enriches the NHS A&E provider-level dataset with external features that may improve forecasting performance.

## External Features Planned
- UK public holidays
- Weather variables
- Population estimates

## Current Focus
This notebook starts by adding **public holiday features** to the monthly NHS A&E dataset.

In [ ]:
# Imports
import os
import json
import requests
import numpy as np
import pandas as pd

In [2]:
# Display Options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [3]:
# Seth Path
processed_folder = "../data/processed"
external_folder = "../data/external"

os.makedirs(processed_folder, exist_ok=True)
os.makedirs(external_folder, exist_ok=True)

input_file = os.path.join(processed_folder, "nhs_ae_full_2020_2026.csv")
bank_holidays_file = os.path.join(external_folder, "bank_holidays.csv")
output_file = os.path.join(processed_folder, "nhs_ae_with_external_features.csv")

In [4]:
# Load dataset
df = pd.read_csv(input_file)
df.head()

,period,org_code,parent_org,org_name,aande_attendances_type_1,aande_attendances_type_2,aande_attendances_other_aande_department,attendances_over_4hrs_type_1,attendances_over_4hrs_type_2,attendances_over_4hrs_other_department,patients_who_have_waited_4_12_hs_from_dta_to_admission,patients_who_have_waited_12plus_hrs_from_dta_to_admission,emergency_admissions_via_aande___type_1,emergency_admissions_via_aande___type_2,emergency_admissions_via_aande___other_aande_department,other_emergency_admissions,source_file,aande_attendances_booked_appointments_type_1,aande_attendances_booked_appointments_type_2,aande_attendances_booked_appointments_other_department,attendances_over_4hrs_booked_appointments_type_1,attendances_over_4hrs_booked_appointments_type_2,attendances_over_4hrs_booked_appointments_other_department,total_attendances,total_booked_attendances,total_over_4hrs,total_booked_over_4hrs,total_dta_waits,total_emergency_admissions,over_4hr_rate,booked_over_4hr_rate,admission_rate
0,2020-04-01,RCF,NHS ENGLAND NORTH EAST AND YORKSHIRE,AIREDALE NHS FOUNDATION TRUST,2953,182,7,276,0,0,4,0,1006,0,0,287,April-2020-revised-260421-abc123.csv,0.0,0.0,0.0,0.0,0.0,0.0,3142,0.0,276,0.0,4,1293,0.087842,0.0,0.411521
1,2020-05-01,RCF,NHS ENGLAND NORTH EAST AND YORKSHIRE,AIREDALE NHS FOUNDATION TRUST,3858,262,1,304,0,0,0,0,1190,0,0,404,May-2020-revised-260421-de345.csv,0.0,0.0,0.0,0.0,0.0,0.0,4121,0.0,304,0.0,0,1594,0.073769,0.0,0.386799
2,2020-06-01,RCF,NHS ENGLAND NORTH EAST AND YORKSHIRE,AIREDALE NHS FOUNDATION TRUST,4216,309,7,280,0,0,0,0,1314,0,0,467,June-2020-revised-260421-df453.csv,0.0,0.0,0.0,0.0,0.0,0.0,4532,0.0,280,0.0,0,1781,0.061783,0.0,0.392983
3,2020-07-01,RCF,NHS ENGLAND NORTH EAST AND YORKSHIRE,AIREDALE NHS FOUNDATION TRUST,4668,357,16,316,0,0,2,0,1164,0,0,378,July-2020-revised-270421-cd305.csv,0.0,0.0,0.0,0.0,0.0,0.0,5041,0.0,316,0.0,2,1542,0.062686,0.0,0.305892
4,2020-08-01,RCF,NHS ENGLAND NORTH EAST AND YORKSHIRE,AIREDALE NHS FOUNDATION TRUST,4913,332,11,514,0,0,11,0,1266,0,0,357,August-2020-revised-270421-gh920.csv,0.0,0.0,0.0,0.0,0.0,0.0,5256,0.0,514,0.0,11,1623,0.097793,0.0,0.308790


In [5]:
df.shape

(14529, 32)

In [6]:
df["period"] = pd.to_datetime(df["period"])
df["period"].head()

0   2020-04-01
1   2020-05-01
2   2020-06-01
3   2020-07-01
4   2020-08-01
Name: period, dtype: datetime64[ns]

In [7]:
# Core checks
required_cols = ["org_code", "org_name", "period", "parent_org", "total_attendances"]

missing_cols = [col for col in required_cols if col not in df.columns]
missing_cols

[]

In [8]:
# Validation Summary
print("Rows:", len(df))
print("Unique organisations:", df["org_code"].nunique())
print("Date range:", df["period"].min(), "to", df["period"].max())
print("Unique months:", df["period"].nunique())

Rows: 14529
Unique organisations: 239
Date range: 2020-04-01 00:00:00 to 2026-02-01 00:00:00
Unique months: 71


## Download UK Bank Holidays

Use the GOV.UK bank holidays feed and extract the England and Wales calendar.

In [9]:
url = "https://www.gov.uk/bank-holidays.json"
response = requests.get(url, timeout=30)
response.raise_for_status()

bank_holiday_data = response.json()
bank_holiday_data.keys()

dict_keys(['england-and-wales', 'scotland', 'northern-ireland'])

In [10]:
# England and Wales Holidays
england_wales_holidays = bank_holiday_data["england-and-wales"]["events"]
england_wales_holidays[:3]

[{'title': 'New Year’s Day',
  'date': '2019-01-01',
  'notes': '',
  'bunting': True},
 {'title': 'Good Friday', 'date': '2019-04-19', 'notes': '', 'bunting': False},
 {'title': 'Easter Monday',
  'date': '2019-04-22',
  'notes': '',
  'bunting': True}]

In [11]:
# Convert to DataFrame
bank_holidays = pd.DataFrame(england_wales_holidays)
bank_holidays.head()

,title,date,notes,bunting
0,New Year’s Day,2019-01-01,,True
1,Good Friday,2019-04-19,,False
2,Easter Monday,2019-04-22,,True
3,Early May bank holiday,2019-05-06,,True
4,Spring bank holiday,2019-05-27,,True


In [12]:
# Clean Holiday data
bank_holidays["date"] = pd.to_datetime(bank_holidays["date"])
bank_holidays["year"] = bank_holidays["date"].dt.year
bank_holidays["month"] = bank_holidays["date"].dt.month
bank_holidays["period"] = bank_holidays["date"].dt.to_period("M").dt.to_timestamp()

bank_holidays.head()

,title,date,notes,bunting,year,month,period
0,New Year’s Day,2019-01-01,,True,2019,1,2019-01-01
1,Good Friday,2019-04-19,,False,2019,4,2019-04-01
2,Easter Monday,2019-04-22,,True,2019,4,2019-04-01
3,Early May bank holiday,2019-05-06,,True,2019,5,2019-05-01
4,Spring bank holiday,2019-05-27,,True,2019,5,2019-05-01


In [13]:
#Raw Holiday Table
bank_holidays.to_csv(bank_holidays_file, index=False)
print(f"Saved bank holidays to: {bank_holidays_file}")

Saved bank holidays to: ../data/external/bank_holidays.csv


## Monthly Holiday Features

Because the NHS A&E data is monthly, holiday data is aggregated to the month level.

In [14]:
holiday_monthly = (
    bank_holidays.groupby("period", as_index=False)
    .agg(
        holiday_count_in_month=("title", "count")
    )
)

holiday_monthly["is_holiday_month"] = (
    holiday_monthly["holiday_count_in_month"] > 0
).astype(int)

holiday_monthly.head()

,period,holiday_count_in_month,is_holiday_month
0,2019-01-01,1,1
1,2019-04-01,2,1
2,2019-05-01,2,1
3,2019-08-01,1,1
4,2019-12-01,2,1


In [15]:
bank_holidays["holiday_name"] = bank_holidays["title"].str.lower()

christmas_periods = bank_holidays.loc[
    bank_holidays["holiday_name"].str.contains("christmas|boxing", na=False),
    "period"
].drop_duplicates()

easter_periods = bank_holidays.loc[
    bank_holidays["holiday_name"].str.contains("easter|good friday", na=False),
    "period"
].drop_duplicates()

holiday_monthly["has_christmas_period"] = holiday_monthly["period"].isin(christmas_periods).astype(int)
holiday_monthly["has_easter_period"] = holiday_monthly["period"].isin(easter_periods).astype(int)

holiday_monthly.head()

,period,holiday_count_in_month,is_holiday_month,has_christmas_period,has_easter_period
0,2019-01-01,1,1,0,0
1,2019-04-01,2,1,0,1
2,2019-05-01,2,1,0,0
3,2019-08-01,1,1,0,0
4,2019-12-01,2,1,1,0


## Merge Holiday Features into the NHS Dataset

Join the monthly holiday features to the provider-level monthly dataset using the `period` field.